# T-16 — Scoring Weight Sensitivity Analysis

**Purpose:** Test whether small changes to the two weight pairs in the scoring system
significantly reorder the top opportunities, establishing whether the current values
are defensible or need revision before T-15 (recommendation labels).

**Two weight pairs under analysis:**

| Pair | Baseline | Formula |
|------|----------|---------|
| Pair A | `odi=0.60, evidence=0.40` | `priority_score = (odi_score × A1) + (evidence_robustness × A2)` |
| Pair B | `diversity=0.65, size=0.35` | `evidence_robustness = (source_type_diversity × B1) + (importance × B2)` |

**Method:** Grid search ±0.10 in 0.05 steps for each pair (pairs always sum to 1.0).
For each weight combination, compute Kendall's tau rank correlation vs the baseline
ranking. Tau > 0.90 across all variations = weights are stable.

## 0. Setup

In [1]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

REPO_ROOT = os.path.abspath('..')  # notebook lives in notebooks/
sys.path.insert(0, REPO_ROOT)

import pandas as pd
import numpy as np
from pathlib import Path
from itertools import product
from scipy.stats import kendalltau

from pipeline.chunker import chunk_text
from pipeline.embedder import embed_chunks
from pipeline.clusterer import cluster
from pipeline.odi_scorer import score_clusters, _batch_sentiment, KNOWN_SOURCE_TYPES_COUNT

print('All imports OK')
print(f'Repo root: {REPO_ROOT}')
print(f'KNOWN_SOURCE_TYPES_COUNT = {KNOWN_SOURCE_TYPES_COUNT}  (expected: 6)')

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7899.59it/s]


All imports OK
Repo root: /Users/lucashome/capstone/discovery-lens
KNOWN_SOURCE_TYPES_COUNT = 6  (expected: 6)


## 1. Load and prepare all three datasets

We need raw `chunks`, `embeddings`, and `clusters` for each dataset.
The scoring step is re-run for every weight combination, so we separate
it from the expensive embedding + clustering step.

In [2]:
# ── Loaders (same as T-08 notebook) ─────────────────────────────────────────

def load_txt(path, source_type):
    raw = Path(path).read_text(encoding='utf-8', errors='replace')
    return chunk_text(raw, Path(path).name, source_type)

def load_csv_col(path, text_col, source_type, max_rows=80):
    df = pd.read_csv(path, nrows=max_rows)
    joined = ' '.join(df[text_col].dropna().astype(str).tolist())
    return chunk_text(joined, Path(path).name, source_type)

DATA = Path(REPO_ROOT) / 'data' / 'synthetic'

# ── Lidl ────────────────────────────────────────────────────────────────────
lidl_dir = DATA / 'lidl_plus_app'
lidl_chunks = []
for i in range(1, 6):
    p = lidl_dir / f'interview_lidl_0{i}.txt'
    if p.exists(): lidl_chunks += load_txt(str(p), 'interview')
for i in range(1, 5):
    p = lidl_dir / f'usability_lidl_0{i}.txt'
    if p.exists(): lidl_chunks += load_txt(str(p), 'usability')
lidl_chunks += load_csv_col(lidl_dir / 'lidl_plus_reviews.csv', 'content', 'review', 60)
lidl_chunks += load_csv_col(lidl_dir / 'tickets_lidl.csv', 'text', 'ticket', 40)

# ── Asana ────────────────────────────────────────────────────────────────────
asana_dir = DATA / 'asana'
asana_chunks = []
for i in range(1, 5):
    p = asana_dir / f'interview_asana_0{i}.txt'
    if p.exists(): asana_chunks += load_txt(str(p), 'interview')
for i in range(1, 4):
    p = asana_dir / f'usability_asana_0{i}.txt'
    if p.exists(): asana_chunks += load_txt(str(p), 'usability')
asana_chunks += load_csv_col(asana_dir / 'reviews_asana.csv', 'text', 'review', 60)
asana_chunks += load_csv_col(asana_dir / 'tickets_asana.csv', 'text', 'ticket', 40)
asana_chunks += load_csv_col(asana_dir / 'reddit_asana.csv', 'post_text', 'social', 30)
p = asana_dir / 'sales_notes_asana.txt'
if p.exists(): asana_chunks += load_txt(str(p), 'internal')

# ── Revolut ──────────────────────────────────────────────────────────────────
revolut_chunks = []
p = DATA / 'revolut_app' / 'interview_revolut_01.txt'
if p.exists(): revolut_chunks += load_txt(str(p), 'interview')
revolut_chunks += load_csv_col(DATA / 'revolut' / 'reviews_revolut.csv', 'content', 'review', 80)
revolut_chunks += load_csv_col(DATA / 'revolut' / 'tickets_revolut.csv', 'customer_message', 'ticket', 40)

print(f'Lidl:    {len(lidl_chunks)} chunks')
print(f'Asana:   {len(asana_chunks)} chunks')
print(f'Revolut: {len(revolut_chunks)} chunks')

Lidl:    431 chunks
Asana:   419 chunks
Revolut: 86 chunks


In [3]:
# Embed + cluster — expensive, done once only
print('Embedding and clustering (done once, reused across all weight combinations)...')

lidl_emb      = embed_chunks(lidl_chunks)
lidl_clusters = cluster(lidl_chunks, lidl_emb)
print(f'Lidl:    {len(lidl_clusters)} clusters ✓')

asana_emb      = embed_chunks(asana_chunks)
asana_clusters = cluster(asana_chunks, asana_emb)
print(f'Asana:   {len(asana_clusters)} clusters ✓')

revolut_emb      = embed_chunks(revolut_chunks)
revolut_clusters = cluster(revolut_chunks, revolut_emb)
print(f'Revolut: {len(revolut_clusters)} clusters ✓')

print('\nReady for grid search.')

Embedding and clustering (done once, reused across all weight combinations)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7418.58it/s]


Lidl:    7 clusters ✓
Asana:   4 clusters ✓
Revolut: 3 clusters ✓

Ready for grid search.


## 2. Custom scorer — weight-parameterised

This reimplements the scoring logic from `odi_scorer.py` with the four
weight constants exposed as parameters. The lxyuan sentiment scores are
pre-computed once per dataset (they don't change with weights) and passed
in as `precomputed_sentiments` to avoid running the model 75 times.

In [4]:
def precompute_sentiments(clusters, chunks):
    """
    Run lxyuan once per dataset and return a dict:
        cluster_id → (avg_sentiment, satisfaction)
    These values are fixed regardless of weights.
    """
    chunk_text_map = {c['chunk_id']: c['text'] for c in chunks}
    result = {}
    for cl in clusters:
        texts = [chunk_text_map[cid] for cid in cl['all_chunk_ids'] if cid in chunk_text_map]
        if texts:
            compounds = _batch_sentiment(texts)
            avg_sent = sum(compounds) / len(compounds)
        else:
            avg_sent = 0.0
        result[cl['cluster_id']] = {
            'avg_sentiment': avg_sent,
            'satisfaction': (avg_sent + 1) / 2
        }
    return result


def score_with_weights(clusters, chunks, sentiments,
                       odi_w, evidence_w,
                       diversity_w, size_w):
    """
    Score clusters using the provided weights.
    Returns a dict: cluster_id → priority_score
    (only priority_score varies with weights — that's what we rank on).

    Constraints enforced by the grid builder (not here):
        odi_w + evidence_w == 1.0
        diversity_w + size_w == 1.0
    """
    chunk_source = {c['chunk_id']: c['source_type'] for c in chunks}
    total_chunks = len(chunks)
    scores = {}

    for cl in clusters:
        cid = cl['cluster_id']
        chunk_ids = cl['all_chunk_ids']
        cluster_size = len(chunk_ids)

        importance   = cluster_size / total_chunks
        satisfaction = sentiments[cid]['satisfaction']
        odi_score    = importance * (1 - satisfaction)

        unique_types = len({chunk_source[x] for x in chunk_ids if x in chunk_source})
        diversity    = unique_types / KNOWN_SOURCE_TYPES_COUNT

        evidence_robustness = (diversity * diversity_w) + (importance * size_w)
        priority_score      = (odi_score * odi_w) + (evidence_robustness * evidence_w)

        scores[cid] = priority_score

    return scores


print('Custom scorer defined.')

Custom scorer defined.


In [5]:
# Pre-compute sentiments once per dataset (slow step — ~30s each)
print('Pre-computing sentiments (once per dataset)...')
lidl_sents    = precompute_sentiments(lidl_clusters, lidl_chunks)
print('Lidl done ✓')
asana_sents   = precompute_sentiments(asana_clusters, asana_chunks)
print('Asana done ✓')
revolut_sents = precompute_sentiments(revolut_clusters, revolut_chunks)
print('Revolut done ✓')
print('\nSentiment pre-computation complete. Grid search will be fast.')

Pre-computing sentiments (once per dataset)...
Lidl done ✓
Asana done ✓
Revolut done ✓

Sentiment pre-computation complete. Grid search will be fast.


## 3. Grid search

**Grid:** ±0.10 in 0.05 steps for each pair, both pairs always sum to 1.0.

| Pair A values | Pair B values |
|---|---|
| odi=0.50, evidence=0.50 | diversity=0.55, size=0.45 |
| odi=0.55, evidence=0.45 | diversity=0.60, size=0.40 |
| **odi=0.60, evidence=0.40** ← baseline | **diversity=0.65, size=0.35** ← baseline |
| odi=0.65, evidence=0.35 | diversity=0.70, size=0.30 |
| odi=0.70, evidence=0.30 | diversity=0.75, size=0.25 |

25 combinations × 3 datasets = 75 total runs.
Metric: Kendall's tau vs baseline ranking for each dataset.

In [6]:
# Build the weight grid
PAIR_A_VALUES = [round(v, 2) for v in np.arange(0.50, 0.71, 0.05)]  # odi weight
PAIR_B_VALUES = [round(v, 2) for v in np.arange(0.55, 0.76, 0.05)]  # diversity weight

BASELINE_A = 0.60  # odi_w baseline
BASELINE_B = 0.65  # diversity_w baseline

print('Pair A (odi_w → priority_score):', PAIR_A_VALUES)
print('Pair B (diversity_w → evidence_robustness):', PAIR_B_VALUES)
print(f'Total combinations: {len(PAIR_A_VALUES) * len(PAIR_B_VALUES)}')

datasets = [
    ('Lidl',    lidl_clusters,    lidl_chunks,    lidl_sents),
    ('Asana',   asana_clusters,   asana_chunks,   asana_sents),
    ('Revolut', revolut_clusters, revolut_chunks, revolut_sents),
]

Pair A (odi_w → priority_score): [np.float64(0.5), np.float64(0.55), np.float64(0.6), np.float64(0.65), np.float64(0.7)]
Pair B (diversity_w → evidence_robustness): [np.float64(0.55), np.float64(0.6), np.float64(0.65), np.float64(0.7), np.float64(0.75)]
Total combinations: 25


In [7]:
# Compute baseline rankings once per dataset
baseline_rankings = {}
for name, clusters_, chunks_, sents_ in datasets:
    scores = score_with_weights(
        clusters_, chunks_, sents_,
        odi_w=BASELINE_A, evidence_w=round(1 - BASELINE_A, 2),
        diversity_w=BASELINE_B, size_w=round(1 - BASELINE_B, 2)
    )
    # Rank: highest priority_score = rank 1
    sorted_ids = sorted(scores, key=scores.get, reverse=True)
    baseline_rankings[name] = {cid: rank for rank, cid in enumerate(sorted_ids)}
    print(f'{name} baseline ranking: {sorted_ids}')

print('\nBaseline rankings computed.')

Lidl baseline ranking: [1, 2, 5, 0, 6, 3, 4]
Asana baseline ranking: [1, 3, 0, 2]
Revolut baseline ranking: [0, 2, 1]

Baseline rankings computed.


In [8]:
# Run the full grid
rows = []

for odi_w, diversity_w in product(PAIR_A_VALUES, PAIR_B_VALUES):
    evidence_w = round(1 - odi_w, 2)
    size_w     = round(1 - diversity_w, 2)
    is_baseline = (odi_w == BASELINE_A and diversity_w == BASELINE_B)

    for name, clusters_, chunks_, sents_ in datasets:
        scores = score_with_weights(
            clusters_, chunks_, sents_,
            odi_w=odi_w, evidence_w=evidence_w,
            diversity_w=diversity_w, size_w=size_w
        )
        sorted_ids   = sorted(scores, key=scores.get, reverse=True)
        variant_rank = [sorted_ids.index(cid) for cid in sorted(baseline_rankings[name], key=baseline_rankings[name].get)]
        base_rank    = list(range(len(sorted_ids)))

        tau, _ = kendalltau(base_rank, variant_rank)

        rows.append({
            'dataset':     name,
            'odi_w':       odi_w,
            'evidence_w':  evidence_w,
            'diversity_w': diversity_w,
            'size_w':      size_w,
            'tau':         round(tau, 4),
            'baseline':    is_baseline,
        })

grid_df = pd.DataFrame(rows)
print(f'Grid complete: {len(grid_df)} rows')
print(f'Baseline rows (tau should = 1.0): {grid_df[grid_df.baseline].tau.tolist()}')

Grid complete: 75 rows
Baseline rows (tau should = 1.0): [1.0, 1.0, 1.0]


## 4. Results — stability analysis

In [9]:
# ── Global stability summary ─────────────────────────────────────────────────
non_baseline = grid_df[~grid_df.baseline]

print('=== Global stability summary ===')
print(f'Total variant runs (excl. baseline): {len(non_baseline)}')
print(f'Min tau across all variants:         {non_baseline.tau.min():.4f}')
print(f'Mean tau across all variants:        {non_baseline.tau.mean():.4f}')
print(f'% of runs with tau > 0.90:           {(non_baseline.tau > 0.90).mean()*100:.1f}%')
print(f'% of runs with tau > 0.80:           {(non_baseline.tau > 0.80).mean()*100:.1f}%')
print()

threshold = 0.90
unstable = non_baseline[non_baseline.tau < threshold]
if unstable.empty:
    print(f'✅ All variants tau > {threshold} — weights are STABLE')
else:
    print(f'⚠️  {len(unstable)} variants below tau={threshold} — weights show INSTABILITY')
    print(unstable[['dataset','odi_w','diversity_w','tau']].to_string(index=False))

=== Global stability summary ===
Total variant runs (excl. baseline): 72
Min tau across all variants:         0.9048
Mean tau across all variants:        0.9947
% of runs with tau > 0.90:           100.0%
% of runs with tau > 0.80:           100.0%

✅ All variants tau > 0.9 — weights are STABLE


In [10]:
# ── Per-dataset stability ────────────────────────────────────────────────────
print('=== Per-dataset min tau (worst case per dataset) ===')
for name in ['Lidl', 'Asana', 'Revolut']:
    sub = non_baseline[non_baseline.dataset == name]
    print(f'{name:<10}  min tau={sub.tau.min():.4f}  '
          f'mean tau={sub.tau.mean():.4f}  '
          f'all > 0.90: {(sub.tau > 0.90).all()}')

=== Per-dataset min tau (worst case per dataset) ===
Lidl        min tau=0.9048  mean tau=0.9841  all > 0.90: True
Asana       min tau=1.0000  mean tau=1.0000  all > 0.90: True
Revolut     min tau=1.0000  mean tau=1.0000  all > 0.90: True


In [11]:
# ── Which pair drives instability? ──────────────────────────────────────────
# Fix each pair at baseline while varying the other
print('=== Pair A isolated (diversity_w fixed at baseline 0.65) ===')
pair_a_iso = non_baseline[non_baseline.diversity_w == BASELINE_B]
print(pair_a_iso.groupby('odi_w')['tau'].agg(['min','mean']).round(4))

print()
print('=== Pair B isolated (odi_w fixed at baseline 0.60) ===')
pair_b_iso = non_baseline[non_baseline.odi_w == BASELINE_A]
print(pair_b_iso.groupby('diversity_w')['tau'].agg(['min','mean']).round(4))

=== Pair A isolated (diversity_w fixed at baseline 0.65) ===
          min    mean
odi_w                
0.50   0.9048  0.9683
0.55   1.0000  1.0000
0.65   1.0000  1.0000
0.70   1.0000  1.0000

=== Pair B isolated (odi_w fixed at baseline 0.60) ===
             min  mean
diversity_w           
0.55         1.0   1.0
0.60         1.0   1.0
0.70         1.0   1.0
0.75         1.0   1.0


In [12]:
# ── Full grid heatmap — min tau per (odi_w, diversity_w) across all datasets
pivot = non_baseline.groupby(['odi_w', 'diversity_w'])['tau'].min().unstack()
print('=== Heatmap: min tau across all 3 datasets per weight combination ===')
print('(rows = odi_w, cols = diversity_w, values = min tau across datasets)')
print()
print(pivot.to_string())
print()
print('Baseline: odi_w=0.60, diversity_w=0.65')
print('Values < 0.90 indicate unstable weight combinations.')

=== Heatmap: min tau across all 3 datasets per weight combination ===
(rows = odi_w, cols = diversity_w, values = min tau across datasets)

diversity_w    0.55    0.60    0.65  0.70  0.75
odi_w                                          
0.50         0.9048  0.9048  0.9048   1.0   1.0
0.55         0.9048  1.0000  1.0000   1.0   1.0
0.60         1.0000  1.0000     NaN   1.0   1.0
0.65         1.0000  1.0000  1.0000   1.0   1.0
0.70         1.0000  1.0000  1.0000   1.0   1.0

Baseline: odi_w=0.60, diversity_w=0.65
Values < 0.90 indicate unstable weight combinations.


In [13]:
# ── Top-1 stability — does #1 ranked opportunity ever change? ────────────────
# For T-15 / D-02, the most important question is whether the top opportunity
# flips to a different cluster under any weight combination.

print('=== Top-1 stability (does the #1 ranked cluster ever change?) ===')
for name, clusters_, chunks_, sents_ in datasets:
    baseline_top1 = min(baseline_rankings[name], key=baseline_rankings[name].get)
    top1s = set()
    for odi_w, diversity_w in product(PAIR_A_VALUES, PAIR_B_VALUES):
        scores = score_with_weights(
            clusters_, chunks_, sents_,
            odi_w=odi_w, evidence_w=round(1-odi_w, 2),
            diversity_w=diversity_w, size_w=round(1-diversity_w, 2)
        )
        top1s.add(max(scores, key=scores.get))
    stable = len(top1s) == 1
    mark = '✅' if stable else '⚠️ '
    print(f'{name:<10} baseline #1 = cluster {baseline_top1}  '
          f'unique #1s across grid: {sorted(top1s)}  {mark}')

=== Top-1 stability (does the #1 ranked cluster ever change?) ===
Lidl       baseline #1 = cluster 1  unique #1s across grid: [1, 2]  ⚠️ 
Asana      baseline #1 = cluster 1  unique #1s across grid: [1]  ✅
Revolut    baseline #1 = cluster 0  unique #1s across grid: [0]  ✅


## 5. Decision

Interpret the results using this decision tree:

```
All tau > 0.90? ──YES──► Baseline weights are STABLE. Confirm as-is.
                          Log: "weights defensible — no change needed"
      │
      NO
      │
      ▼
Which pair drives the drop?
  Pair A (odi/evidence)? ──► ODI vs evidence tradeoff is sensitive.
                              Check: is high diversity_w masking unmet needs?
                              Consider: raise odi_w to 0.65 or 0.70.
  Pair B (diversity/size)? ──► Source diversity weight is sensitive.
                                Check: small datasets with 1 source type collapsing.
                                Consider: lower diversity_w to 0.60.
  Both pairs? ──► Priority score is under-determined.
                  Consider: simplify — drop evidence_robustness from priority_score,
                  keep as standalone UI signal only.
```

In [14]:
# Print a clean summary for decisions.md
print('=== SUMMARY FOR decisions.md ===')
print()
print('Baseline weights tested:')
print(f'  Pair A: odi_w={BASELINE_A}, evidence_w={round(1-BASELINE_A,2)}')
print(f'  Pair B: diversity_w={BASELINE_B}, size_w={round(1-BASELINE_B,2)}')
print()
print('Grid: ±0.10 in 0.05 steps per pair (25 combinations × 3 datasets = 75 runs)')
print()
for name in ['Lidl', 'Asana', 'Revolut']:
    sub = non_baseline[non_baseline.dataset == name]
    print(f'  {name}: min tau={sub.tau.min():.4f}, mean tau={sub.tau.mean():.4f}')
print()
all_stable = (non_baseline.tau > 0.90).all()
verdict = 'STABLE — baseline weights confirmed' if all_stable else 'UNSTABLE — see heatmap for which combinations to avoid'
print(f'Verdict: {verdict}')
print()
print('Pair A isolated min tau:', pair_a_iso.tau.min().round(4))
print('Pair B isolated min tau:', pair_b_iso.tau.min().round(4))
culprit = []
if pair_a_iso.tau.min() < 0.90: culprit.append('Pair A (odi/evidence)')
if pair_b_iso.tau.min() < 0.90: culprit.append('Pair B (diversity/size)')
print('Instability driver(s):', ', '.join(culprit) if culprit else 'None — both pairs stable')

=== SUMMARY FOR decisions.md ===

Baseline weights tested:
  Pair A: odi_w=0.6, evidence_w=0.4
  Pair B: diversity_w=0.65, size_w=0.35

Grid: ±0.10 in 0.05 steps per pair (25 combinations × 3 datasets = 75 runs)

  Lidl: min tau=0.9048, mean tau=0.9841
  Asana: min tau=1.0000, mean tau=1.0000
  Revolut: min tau=1.0000, mean tau=1.0000

Verdict: STABLE — baseline weights confirmed

Pair A isolated min tau: 0.9048
Pair B isolated min tau: 1.0
Instability driver(s): None — both pairs stable
